# Summary
Script to run against the MSA and CNAF files that have gone through cleaning/formatting process

This notebook merges benefs and sets the default column values such as:
- exercice_id, uuid_doc, zrr, qpv, a_valider, refuser, created_at, updated_at

Unique codes are also generated for each of the rows and assigned to column "id_psp"

In [ ]:
import os
import pandas as pd
import numpy as np
import csv

from dotenv import load_dotenv

In [ ]:
load_dotenv()

cnaf_export_2026_filepath = os.environ['DB_CNAF_EXPORT_2026']
msa_export_2026_filepath = os.environ['DB_MSA_EXPORT_2026']

final_db_export_output_filepath = os.environ['FINAL_DB_EXPORT_2026']

In [ ]:
# keep_default_na is necessary otherwise string such as "NA" is considered as NaN...
df_cnaf_2026 = pd.read_csv(cnaf_export_2026_filepath, sep=';', encoding='utf-8', dtype=str, keep_default_na=False, quoting=csv.QUOTE_ALL)
df_msa_2026= pd.read_csv(msa_export_2026_filepath, sep=';', encoding='utf-8', dtype=str, keep_default_na=False, quoting=csv.QUOTE_ALL)

In [ ]:
assert(len(df_cnaf_2026[df_cnaf_2026['prenom'].isna() | df_cnaf_2026['prenom'].isna()]) == 0)
assert(len(df_msa_2026[df_msa_2026['prenom'].isna() | df_msa_2026['prenom'].isna()]) == 0)

In [ ]:
merged_df_from_2026 = pd.concat([df_cnaf_2026, df_msa_2026], ignore_index=True).reset_index(drop=True)

In [ ]:
merged_df_from_2026['date_naissance_to_compare'] = pd.to_datetime(merged_df_from_2026['date_naissance']).dt.date

timestamp_with_custom_tz = pd.Timestamp.now(tz='Europe/Paris')
merged_df_from_2026[['created_at', 'updated_at']] = timestamp_with_custom_tz

In [ ]:
final_df = merged_df_from_2026

In [ ]:
# Add missing default column needed to production data
exercice_2026 = 5

final_df['exercice_id'] = exercice_2026
final_df['uuid_doc'] = np.NaN
final_df[['zrr', 'qpv', 'a_valider', 'refuser']] = False

In [ ]:
# Unique codes generation
import random
import string
import datetime

current_date = datetime.datetime.now()
current_year = str(current_date.year)[-2:]

def get_characters_set(size = 4):
    return ''.join(random.choices([c for c in string.ascii_uppercase if c not in 'OI'], k=size))

def generate_code():
    return f"{current_year}-{get_characters_set(4)}-{get_characters_set(4)}"

# init set of codes with existing
unique_codes = set()

# init current_code count
current_codes_count = len(unique_codes)

while len(unique_codes) < len(final_df):
    unique_codes.add(generate_code())

# Ensure we have generated codes for all the rows
assert len(unique_codes) == len(final_df)

In [ ]:
# Assign generated code for production data
final_df['id_psp'] = list(unique_codes)

In [ ]:
print(f"{len(merged_df_from_2026)} benefs from 2026 (msa + cnaf)")
print(f"{len(merged_df_from_2026[merged_df_from_2026['genre'] == 'M'])} M benefs from 2026")
print(f"{len(merged_df_from_2026[merged_df_from_2026['genre'] == 'F'])} F benefs from 2026")

In [ ]:
print(f"{len(unique_codes)} unique codes generated for this year")
print(f"{len(final_df)} benefs from 2026 (msa + cnaf)")
print(f"{len(final_df[final_df['genre'] == 'M'])} M benefs from 2026")
print(f"{len(final_df[final_df['genre'] == 'F'])} F benefs from 2026")

In [ ]:
print(f"{len(merged_df_from_2026[merged_df_from_2026['situation'] == 'AAH'])} AAH benefs from 2026 (msa + cnaf)")
print(f"{len(final_df[final_df['situation'] == 'AAH'])} AAH benefs in final (msa + cnaf)")

In [ ]:
print(f"{len(merged_df_from_2026[merged_df_from_2026['situation'] == 'jeune'])} jeune benefs from 2026 (msa + cnaf)")
print(f"{len(final_df[final_df['situation'] == 'jeune'])} jeune benefs in final (msa + cnaf)")

In [ ]:
final_df.drop(columns=['date_naissance_to_compare'], inplace=True)

In [ ]:
mask_jeune = final_df['situation'] == 'jeune'
mask_caf = final_df['organisme'] == 'CAF'
len(final_df[mask_jeune & mask_caf])

In [ ]:
final_df.to_csv(final_db_export_output_filepath, sep=';', index=False, encoding='utf-8')